# Лаба 2 — сравнение RAG

Читаем последний завершённый запуск. Модели не вызываются. Эталоны и метрики — на уровне страниц PDF; k=5 и k=10.


In [ ]:
from pathlib import Path
import csv
import json
from statistics import mean
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'questions.json').exists())
candidates = sorted((ROOT / 'results').glob('*/metadata.json'), reverse=True)
RUN = next((p.parent for p in candidates if json.loads(p.read_text(encoding='utf-8'))['status'] == 'completed'), None)
if RUN is None:
    raise FileNotFoundError('Сначала выполните source/evaluate.py')
metadata = json.loads((RUN / 'metadata.json').read_text(encoding='utf-8'))
rows = list(csv.DictReader((RUN / 'summary.csv').open(encoding='utf-8-sig')))
def table(rows, columns):
    text = '| ' + ' | '.join(columns) + ' |\n|' + '|'.join('---' for _ in columns) + '|\n'
    text += '\n'.join('| ' + ' | '.join(str(r.get(c, '')).replace('|', '/') for c in columns) + ' |' for r in rows)
    display(Markdown(text))
print('Запуск:', RUN.name)
table(rows, ['variant', 'chunks', 'recall@5', 'precision@5', 'mrr@5', 'recall@10', 'precision@10', 'mrr@10'])
table(rows, ['variant', 'index_build_s', 'load_s', 'index_bytes', 'retrieval_s_mean', 'qa_s_mean'])
print('Предварительный выбор по retrieval:', metadata['best_retrieval_variant'])


## Примеры и QA

По одному ответу на первый вопрос каждой конфигурации. Полная проверка — в `qa_review.csv`. Оценки: правильность ответа и подтверждение цитатами; пустые поля не считаются нулём.


In [ ]:
runs = [json.loads(line) for line in (RUN / 'runs.jsonl').read_text(encoding='utf-8').splitlines()]
for variant in metadata['variants']:
    row = next(r for r in runs if r['variant'] == variant)
    print(variant, row['question'])
    print(row.get('answer', 'QA не запускалась'), '\nЭталон:', row['expected_answer'])
    print('Цитаты:', [(c['number'], c['page']) for c in row.get('citations', [])], '\n')
path = RUN / 'qa_review.csv'
if path.exists():
    reviews = list(csv.DictReader(path.open(encoding='utf-8-sig')))
    scores = []
    for variant in metadata['variants']:
        group = [r for r in reviews if r['variant'] == variant]
        row = {'variant': variant, 'answers': len(group)}
        for column in ['answer_correct_0_or_1', 'citations_support_answer_0_or_1']:
            values = [r[column].strip() for r in group if r[column].strip()]
            assert all(v in ('0', '1') for v in values), 'Оценки должны быть 0 или 1'
            row[column] = f'{mean(map(int, values)):.3f} ({len(values)}/{len(group)} проверено)' if values else 'Не проверено'
        scores.append(row)
    table(scores, ['variant', 'answers', 'answer_correct_0_or_1', 'citations_support_answer_0_or_1'])


## Выводы

После запуска сравнить качество retrieval, QA и время. Указать лучшую конфигурацию и примеры ошибок. При отсутствии ручных оценок вывод о качестве ответов делать рано. Эти 15 вопросов служат для подбора конфигурации, независимый тест не проводился.
